# Options from Zero · Episode 7: Many steps, and American options

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. Steps set by volatility · 4. A two-step tree · 5. More steps · 6. An American put · 7. The value of the right · 8. Calls are different · 9. Check against QuantLib · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 7,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

If interest rates rose, would $q$ go up or down? Up: $q = (e^{rT} - d)/(u - d)$ grows with $r$.

In [ ]:
q_at = lambda rate: (math.exp(rate * T) - 0.8) / (1.2 - 0.8)
out["answer"] = {"q_now": q_at(r), "q_at_5": q_at(0.05), "rate_hi_pct": 5.0, "q_now_pct": q_at(r) * 100, "q_hi_pct": q_at(0.05) * 100}
out["answer"]

## 3. Steps set by volatility

A real tree has many small steps. Cox, Ross and Rubinstein (1979) size them from volatility: each step multiplies the share by $u = e^{\sigma\sqrt{\Delta t}}$ or $d = 1/u$. Episode 9 shows why the square root. With 20% volatility and two six-month steps:

In [ ]:
def tree_nodes(steps, call=True, american=False, strike=K):
    '''Every node of a CRR tree with its option value; flags nodes where exercising early beats holding.'''
    dt = T / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    q = (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    values = {}
    for k in range(steps + 1):
        values[(steps, k)] = (float(pay(S * u ** k * d ** (steps - k), strike)), False)
    for t in range(steps - 1, -1, -1):
        for k in range(t + 1):
            price = S * u ** k * d ** (t - k)
            hold = math.exp(-r * dt) * (q * values[(t + 1, k + 1)][0] + (1 - q) * values[(t + 1, k)][0])
            ex = float(pay(price, strike))
            early = american and ex > hold + 1e-12
            values[(t, k)] = (max(hold, ex) if american else hold, early)
    nodes = [{"t": t, "k": k, "price": S * u ** k * d ** (t - k), "value": v, "highlight": e}
             for (t, k), (v, e) in sorted(values.items())]
    return nodes, u, d, q

two, u2, d2, q2 = tree_nodes(2)
out["steps"] = {"vol_pct": vol * 100, "dt": 0.5, "u": u2, "d": d2, "q": q2, "up_price": S * u2, "down_price": S * d2}
out["steps"]

## 4. A two-step tree

Prices go forward; values come backward. At expiry the call pays its payoff; one step earlier each node is the discounted $q$-average of the two nodes after it; and so on back to today.

In [ ]:
out["two"] = {"nodes": two, "price": two[0]["value"], "top": S * u2 ** 2, "bottom": S * d2 ** 2,
              "top_payoff": max(S * u2 ** 2 - K, 0), "mid_up_value": [n for n in two if n["t"] == 1 and n["k"] == 1][0]["value"]}
assert abs(out["two"]["price"] - binomial_tree(S, K, T, r, vol, 2)) < 1e-12
{k: v for k, v in out["two"].items() if k != "nodes"}

## 5. More steps

With more steps the price settles down. (Episode 10 shows what it settles to.)

In [ ]:
ns = list(range(1, 101))
prices = [binomial_tree(S, K, T, r, vol, n) for n in ns]
out["converge"] = {"steps": ns, "price": prices, "p10": prices[9], "p50": prices[49], "p100": prices[99],
                   "p500": binomial_tree(S, K, T, r, vol, 500),
                   "ticks": [{"x": float(x), "label": str(x)} for x in (1, 20, 40, 60, 80, 100)]}
{k: v for k, v in out["converge"].items() if k.startswith("p")}

## 6. An American put

A **European** option can be exercised only at expiry; an **American** one on any day up to expiry. ASX stock options are usually American, index options European (ASX, *Options course, Module 2*). With an American put, check at every node whether exercising now (\$105 − share price, in cash today, which then earns interest) beats holding on. A three-step tree:

In [ ]:
am3, u3, d3, q3 = tree_nodes(3, call=False, american=True)
eu3, *_ = tree_nodes(3, call=False, american=False)
early = [n for n in am3 if n["highlight"]]
out["put3"] = {"nodes": am3, "american": am3[0]["value"], "european": eu3[0]["value"], "early_nodes": len(early),
               "early_prices": [n["price"] for n in early], "early_values": [n["value"] for n in early]}
{k: v for k, v in out["put3"].items() if k != "nodes"}

## 7. The value of the right

With 500 steps: the American put is worth more than the European one by the value of being allowed to exercise early. The **exercise boundary** is the share price below which exercising now is best; it rises as expiry nears.

In [ ]:
def boundary(steps=500):
    dt = T / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    q = (math.exp(r * dt) - d) / (u - d)
    prices = S * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = put_payoff(prices, K)
    bnd = {}
    for i in range(steps, 0, -1):
        prices = prices[1:] / u
        hold = math.exp(-r * dt) * (q * values[1:] + (1 - q) * values[:-1])
        ex = put_payoff(prices, K)
        hit = prices[(ex > hold) & (ex > 0)]
        bnd[round((i - 1) * dt, 6)] = float(hit.max()) if len(hit) else None
        values = np.maximum(hold, ex)
    return float(values[0]), bnd

am500, bnd = boundary()
eu500 = binomial_tree(S, K, T, r, vol, 500, call=False)
rows = [{"months_left": 12 - int(round(t * 12)), "exercise_below": bnd[t]} for t in (0.25, 0.5, 0.75)]
out["put500"] = {"american": am500, "european": eu500, "bs_european": black_scholes(S, K, T, r, vol, False),
                 "early_premium": am500 - eu500, "rows": rows,
                 "b9": rows[0]["exercise_below"], "b6": rows[1]["exercise_below"], "b3": rows[2]["exercise_below"]}
{k: v for k, v in out["put500"].items()}

## 8. Calls are different

On a share that pays no dividends, exercising a call early never pays: you'd hand over \$105 sooner and give up the insurance the call still carries. The American call is worth exactly the European one.

In [ ]:
am_call = binomial_tree(S, K, T, r, vol, 500, call=True, american=True)
eu_call = binomial_tree(S, K, T, r, vol, 500, call=True, american=False)
out["calls"] = {"american": am_call, "european": eu_call, "diff": am_call - eu_call}
assert abs(am_call - eu_call) < 1e-12
out["calls"]

## 9. Check against QuantLib

QuantLib's CRR engine uses the same $u$ and $d$ but a slightly different up-probability, $\tfrac12 + \tfrac12 (r - \sigma^2/2)\sqrt{\Delta t}/\sigma$ (a linearised version of ours). Both converge to the same price; with QuantLib's probability our hand tree matches QuantLib exactly.

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
res = {}
for name, kind, exercise, am in (("euro_call", ql.Option.Call, ql.EuropeanExercise(expiry), False),
                                 ("amer_put", ql.Option.Put, ql.AmericanExercise(today, expiry), True)):
    opt = ql.VanillaOption(ql.PlainVanillaPayoff(kind, K), exercise)
    opt.setPricingEngine(ql.BinomialCRRVanillaEngine(process, 500))
    hand = binomial_tree(S, K, T, r, vol, 500, call=(kind == ql.Option.Call), american=am, quantlib_p=True)
    res[name] = {"quantlib": opt.NPV(), "hand_quantlib_p": hand, "hand_textbook": binomial_tree(S, K, T, r, vol, 500, call=(kind == ql.Option.Call), american=am)}
    assert abs(opt.NPV() - hand) < 1e-9
out["quantlib"] = res
res

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")